# 05e — DepthFormer-RGB (Swin-Tiny + UPerNet + PPM)

**Paper**: Ma et al. — *DepthFormer: Depth-Enhanced Transformer Network for Semantic Segmentation of the Martian Surface from Rover Images*  
DOI: `10.1029/2024EA003812` — Earth and Space Science 12, e2024EA003812 (2025)  

**Variante implementada**: DepthFormer-**RGB** — sin canal de profundidad.  

**Justificación académica**: AI4MARS no dispone de depth maps. Esta variante replica exactamente la rama de ablación que el propio paper reporta en su Tabla 3 (`Swin Transformer base` vs `DepthFormer`), lo cual hace la comparación académicamente válida. El paper reporta que el Swin base (sin profundidad) obtiene mIoU=73.58 en DepthMars; nuestra implementación equivale funcionalmente a eso pero sobre AI4MARS.

**Arquitectura**:  
- **Encoder**: Swin-Tiny jerárquico con `img_size=256` (timm, pretrained) — sin interpolación posicional  
- **Decoder**: UPerNet inspirado en UPerNet (Xiao et al., 2018) con FPN top-down  
- **PPM**: Pyramid Pooling Module de PSPNet al final del encoder  
- **Input**: `[B, 3, 256, 256]` — sin canal de depth

## 0. Setup

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/ai4mars_DL-v3')
    sys.path.append(str(PROJECT_ROOT / 'notebooks'))
else:
    PROJECT_ROOT = Path('__file__').resolve().parent.parent
    if not (PROJECT_ROOT / 'processed').exists():
        PROJECT_ROOT = Path.cwd().parent

print(f'ROOT: {PROJECT_ROOT} | existe: {PROJECT_ROOT.exists()}')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
import matplotlib.pyplot as plt

from mars_utils import (
    set_seed, load_norm_stats, load_split,
    build_dataloaders,
    train_one_epoch, evaluate, train_model, run_multi_seed,
    append_benchmark_results, plot_best_seed_curves,
    print_summary_table, visualize_predictions, count_parameters,
    NUM_CLASSES, IGNORE_INDEX, SEEDS
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Configuración

In [ ]:
MODEL_NAME   = 'DepthFormer-RGB'
FAST_SUBSET  = False   # ← True para prueba rápida (~2 min/seed)

LR           = 1e-4
WEIGHT_DECAY = 0.01
MAX_EPOCHS   = 80
PATIENCE     = 10
BATCH_SIZE   = 16

# class weights: soil=1.0, bedrock=0.8, sand=2.0, big_rock=8.0
CLASS_WEIGHTS = torch.tensor([1.0, 0.8, 2.0, 8.0], dtype=torch.float32).to(DEVICE)

print(f'Modo: {"FAST SUBSET" if FAST_SUBSET else "PRODUCCIÓN"}')
print('Swin-Tiny configurado con img_size=256 — sin interpolación posicional')

## 2. Datos

In [ ]:
df_train, df_val, df_gold = load_split()
mean, std = load_norm_stats()

train_ids = set(df_train['stem'].tolist())
gold_ids  = set(df_gold['stem'].tolist())
assert len(train_ids & gold_ids) == 0, '⚠️ DATA LEAKAGE detectado'
print(f'✅ Train: {len(df_train)} | Val: {len(df_val)} | Gold: {len(df_gold)}')

## 3. Arquitectura — DepthFormer-RGB

### 3.1 Pyramid Pooling Module (PPM)

In [ ]:
class PyramidPoolingModule(nn.Module):
    """
    Pyramid Pooling Module de PSPNet (Zhao et al., 2017).
    Captura información contextual a múltiples escalas mediante
    pooling global con ventanas de tamaño creciente (1,2,3,6).
    
    Usado en el último stage del encoder (antes del decoder FPN).
    """
    def __init__(self, in_channels: int, pool_sizes: tuple = (1, 2, 3, 6)):
        super().__init__()
        mid_ch = in_channels // len(pool_sizes)
        self.stages = nn.ModuleList([
            nn.Sequential(
                nn.AdaptiveAvgPool2d(size),
                nn.Conv2d(in_channels, mid_ch, 1, bias=False),
                nn.BatchNorm2d(mid_ch),
                nn.ReLU(inplace=True)
            ) for size in pool_sizes
        ])
        # Fusión: concat original + 4 ramas → proyección
        fused_ch = in_channels + mid_ch * len(pool_sizes)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(fused_ch, in_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        H, W = x.shape[-2:]
        parts = [x]
        for stage in self.stages:
            pooled = stage(x)
            parts.append(
                F.interpolate(pooled, size=(H, W), mode='bilinear', align_corners=False)
            )
        return self.bottleneck(torch.cat(parts, dim=1))

In [ ]:
class FPNDecoder(nn.Module):
    """
    Decoder top-down tipo Feature Pyramid Network.
    Fusiona features de distintos niveles del encoder:
    - Nivel más profundo: ya procesado por PPM
    - Niveles más superficiales: fusionados top-down
    
    Inspirado en UPerNet (Xiao et al., 2018).
    """
    def __init__(self, in_channels_list: list, out_ch: int = 256):
        super().__init__()
        # Proyectar cada nivel lateral a out_ch
        self.laterals = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(ch, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
            ) for ch in in_channels_list
        ])
        # Suavizar tras fusión top-down
        self.smooths = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
            ) for _ in in_channels_list
        ])
        # Fusión final de todos los niveles
        self.fuse = nn.Sequential(
            nn.Conv2d(out_ch * len(in_channels_list), out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )

    def forward(self, features: list):
        """
        features: lista de tensores del encoder [C0, C1, C2, C3],
                  de menor a mayor nivel semántico (C3 es el más profundo).
        """
        # Proyecciones laterales
        laterals = [lat(f) for lat, f in zip(self.laterals, features)]

        # Top-down: fusionar desde el nivel más profundo hacia el más superficial
        for i in range(len(laterals) - 2, -1, -1):
            laterals[i] = laterals[i] + F.interpolate(
                laterals[i + 1], size=laterals[i].shape[-2:],
                mode='bilinear', align_corners=False
            )

        # Suavizar
        outs = [smooth(lat) for smooth, lat in zip(self.smooths, laterals)]

        # Upsample todos al tamaño del nivel más superficial y concatenar
        target = outs[0].shape[-2:]
        outs_up = [
            F.interpolate(o, size=target, mode='bilinear', align_corners=False)
            for o in outs
        ]
        return self.fuse(torch.cat(outs_up, dim=1))

In [ ]:
class DepthFormerRGB(nn.Module):
    """
    DepthFormer-RGB: variante sin canal de profundidad.
    
    Encoder: Swin-Tiny (timm, img_size=256, pretrained)
      Stage 1: 96ch,  H/4×W/4
      Stage 2: 192ch, H/8×W/8
      Stage 3: 384ch, H/16×W/16
      Stage 4: 768ch, H/32×W/32  → PPM → 768ch
    
    Decoder: FPN top-down (256ch internos) → upsample × 4 → cabeza clasificadora
    
    Salida: [B, num_classes, H, W]
    
    Diferencia con DepthFormer original: entrada es [B,3,H,W] en lugar de [B,4,H,W].
    Equivale al ablation 'Swin Transformer base' de la Tabla 3 del paper.
    """
    def __init__(self, num_classes: int = 4, img_size: int = 256):
        super().__init__()

        # Encoder: Swin-Tiny con img_size=256 — evita interpolación posicional
        self.encoder = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=True,
            features_only=True,
            out_indices=(0, 1, 2, 3),
            img_size=img_size       # sobreescribe el img_size del modelo pretrained
        )
        # Canales de Swin-Tiny: [96, 192, 384, 768]
        enc_chs = [96, 192, 384, 768]

        # PPM en el último stage (768ch)
        self.ppm = PyramidPoolingModule(in_channels=enc_chs[-1])

        # Decoder FPN
        self.decoder = FPNDecoder(in_channels_list=enc_chs, out_ch=256)

        # Cabeza clasificadora
        self.head = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
            nn.Conv2d(128, num_classes, 1)
        )

    def forward(self, x):
        H, W = x.shape[-2:]

        # Encoder Swin → lista de features [B, H', W', C] → permutar a [B, C, H', W']
        raw_feats = self.encoder(x)
        feats = [f.permute(0, 3, 1, 2).contiguous() for f in raw_feats]
        # feats[0]: H/4,  96ch
        # feats[1]: H/8,  192ch
        # feats[2]: H/16, 384ch
        # feats[3]: H/32, 768ch

        # PPM en el feature más profundo
        feats[3] = self.ppm(feats[3])

        # Decoder FPN top-down
        dec = self.decoder(feats)   # (B, 256, H/4, W/4)

        # Clasificador + upsample a resolución original
        out = F.interpolate(self.head(dec), size=(H, W),
                            mode='bilinear', align_corners=False)
        return out


def build_model():
    return DepthFormerRGB(num_classes=NUM_CLASSES, img_size=256).to(DEVICE)


# Verificación
_m = build_model()
_m.eval()
with torch.no_grad():
    _x = torch.randn(2, 3, 256, 256).to(DEVICE)
    _out = _m(_x)
    print(f'Forward OK — salida: {_out.shape}')  # (2, 4, 256, 256)

n_params = count_parameters(_m)
print(f'Parámetros entrenables: {n_params:.2f}M  (referencia KB: ~31.58M)')
del _m, _x, _out

## 4. Loss, Optimizer y Scheduler

In [ ]:
def criterion_fn():
    return nn.CrossEntropyLoss(
        weight=CLASS_WEIGHTS,
        ignore_index=IGNORE_INDEX
    )


def optimizer_fn(params):
    return torch.optim.AdamW(
        params, lr=LR, weight_decay=WEIGHT_DECAY
    )


def scheduler_fn(optimizer):
    # PolynomialLR con power=0.9 durante MAX_EPOCHS iteraciones
    return torch.optim.lr_scheduler.PolynomialLR(
        optimizer, total_iters=MAX_EPOCHS, power=0.9
    )


print('✅ Loss: CrossEntropyLoss con class weights [1.0, 0.8, 2.0, 8.0]')
print('  Optimizer: AdamW (lr=1e-4, wd=0.01)')
print('  Scheduler: PolynomialLR (power=0.9, 80 iters)')

## 5. Entrenamiento Multi-Seed

In [ ]:
summary = run_multi_seed(
    model_fn       = build_model,
    df_train       = df_train,
    df_val         = df_val,
    df_gold        = df_gold,
    criterion_fn   = criterion_fn,
    optimizer_fn   = optimizer_fn,
    scheduler_fn   = scheduler_fn,
    model_name     = MODEL_NAME,
    device         = DEVICE,
    mean           = mean,
    std            = std,
    max_epochs     = MAX_EPOCHS,
    patience       = PATIENCE,
    batch_size     = BATCH_SIZE,
    fast_subset    = FAST_SUBSET,
    n_train_fast   = 200,
    n_val_fast     = 50,
)

## 6. Resultados Agregados

In [ ]:
print_summary_table(summary)

In [ ]:
plot_best_seed_curves(summary)

In [ ]:
params_M = count_parameters(build_model())
append_benchmark_results(summary, params_M=params_M)
print('✅ Resultados guardados en results/benchmark_results.csv')

## 7. Visualización Cualitativa

In [ ]:
best_seed  = summary['best_seed']
ckpt_path  = summary['checkpoints'][best_seed]

best_model = build_model()
best_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
best_model.eval()

print(f'Mejor seed: {best_seed} | mIoU gold: {summary["per_seed"][best_seed]["miou_gold"]:.4f}')

visualize_predictions(
    model=best_model, df=df_gold, device=DEVICE,
    mean=mean, std=std, n=5
)

## 8. Nota sobre la variante RGB vs DepthFormer original

El paper original procesa inputs de 4 canales `[R, G, B, Depth]`. Esta implementación usa 3 canales `[R, G, B]` porque AI4MARS no provee depth maps.

| Aspecto | DepthFormer original | Esta implementación |
|---------|---------------------|--------------------|
| Input | [B, 4, H, W] | [B, 3, H, W] |
| Dataset | DepthMars (Zhurong) | AI4MARS (Curiosity) |
| mIoU reportado | 75.99% | — (en evaluación) |
| Ablación comparable | Swin base: 73.58% | Equivalente |

## 9. Resumen Final

| Campo | Valor |
|-------|-------|
| Modelo | DepthFormer-RGB |
| Paper | Ma et al., Earth and Space Science 12 (2025) |
| Encoder | Swin-Tiny (timm, img_size=256, pretrained) |
| Decoder | UPerNet + FPN top-down + PPM |
| Loss | CrossEntropyLoss con class weights |
| Optimizer | AdamW (lr=1e-4, wd=0.01) |
| Scheduler | PolynomialLR (power=0.9) |
| Referencia histórica (2.1k imgs) | mIoU = 0.7609 ± 0.0123 |

---
*Resultados del gold set exportados a `results/benchmark_results.csv`.*